# Behavior-Based Attack Chain Detection Graph

> **Platform:** Microsoft Sentinel Custom Graph | **Domain:** Cross-Domain Fusion
> **Data Sources:** SentinelBehaviorInfo, SentinelBehaviorEntities, AlertInfo, AlertEvidence, ThreatIntelIndicators, BehaviorAnalytics
> **Nodes:** 9 (Behavior, Tactic, Technique, IP, User, AWSResource, Device, Alert, ThreatIndicator)
> **Edges:** 10 (BehaviorMappedToTactic, BehaviorMappedToTechnique, BehaviorInvolvesIP, BehaviorInvolvesUser, BehaviorInvolvesAWSResource, BehaviorInvolvesDevice, IPTriggeredAlert, IPMatchesTI, AlertInvolvesIP, UserHasAlert)

Models the relationships between behavioral detections, MITRE ATT&CK tactics and techniques, network entities, cloud resources, alerts, and threat intelligence. Enables multi-hop graph traversals using GQL to answer questions that are difficult or impossible to express with tabular KQL queries alone.

## Environment & Configuration

Import required packages and configure the Sentinel Data Lake connection.


In [ ]:
# SDK compatibility check — ensures minimum sentinel_graph version


# 📅 OPTIONAL: Adjust the lookback window (default: 7 days). Use 1 for IR, 14 for hunting, 30+ for historical.
TIME_WINDOW_DAYS = 1

# =============================================================================
# Behavior-Based Attack Chain Detection Graph
# Nodes: Behavior, Tactic, Technique, IP, User, AWSResource, Device, Alert, ThreatIndicator
# Edges: 10 relationship types connecting entities through behaviors
# Data Sources: SentinelBehaviorInfo, SentinelBehaviorEntities,
# AlertInfo, AlertEvidence, ThreatIntelIndicators, BehaviorAnalytics
# =============================================================================

from sentinel_lake.providers import MicrosoftSentinelProvider
from sentinel_graph import GraphSpecBuilder, Graph
from pyspark.sql.functions import (
 col, lit, coalesce, explode, concat_ws, first,
 from_json, trim, regexp_replace, get_json_object,
 hour, dayofweek, when, expr, row_number
)
import pyspark.sql.functions as F
from pyspark.sql.types import ArrayType, StringType
from pyspark.sql.window import Window

# ⚠️ REQUIRED: Set your Microsoft Sentinel workspace name before running this notebook
workspace_name = "<YOUR_WORKSPACE_NAME>" # ← Replace with your Sentinel workspace name
lake_provider = MicrosoftSentinelProvider(spark)

## Data Ingestion

Read source tables from the Sentinel Data Lake and persist to Spark memory.


In [ ]:
# =============================================================================
# STEP 1: Load all data tables and persist to Spark memory
# TimeGenerated filters limit scope to the most recent 14 days (30 days for TI)
# =============================================================================

# --- Behavior detections (one row per behavioral detection) ---
behavior_df = lake_provider.read_table('SentinelBehaviorInfo', workspace_name)
behavior_df = behavior_df.df if hasattr(behavior_df, 'df') else behavior_df
behavior_df = behavior_df.where(
 col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS")
).persist()

# --- Behavior entities (multiple entity rows per behavior - IPs, Users, Resources) ---
entities_df = lake_provider.read_table('SentinelBehaviorEntities', workspace_name)
entities_df = entities_df.df if hasattr(entities_df, 'df') else entities_df
entities_df = entities_df.where(
 col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS")
).persist()

print(f'📊 Behaviors loaded (14d): {behavior_df.count()}')
print(f'📊 Entities loaded (14d): {entities_df.count()}')

# --- Security alerts from Microsoft Defender products ---
alert_info_df = lake_provider.read_table('AlertInfo', workspace_name)
alert_info_df = alert_info_df.df if hasattr(alert_info_df, 'df') else alert_info_df
alert_info_df = (alert_info_df
 .where(col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
 .select('AlertId', 'Title', 'Category', 'Severity',
 'ServiceSource', 'DetectionSource', 'TimeGenerated')
).persist()
print(f'📊 Alerts loaded (14d): {alert_info_df.count()}')

# --- Alert evidence — load IP, Device, and User entities for graph edge construction ---
alert_evidence_raw = lake_provider.read_table('AlertEvidence', workspace_name)
alert_evidence_raw = alert_evidence_raw.df if hasattr(alert_evidence_raw, 'df') else alert_evidence_raw
alert_evidence_raw = alert_evidence_raw.where(
 col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS")
).persist()

# IP entities (existing)
alert_evidence_df = (alert_evidence_raw
 .where(
 (col('EntityType') == 'Ip') &
 col('RemoteIP').isNotNull() & (col('RemoteIP') != '')
 )
 .select('AlertId', 'RemoteIP', 'EvidenceRole', 'TimeGenerated')
)
print(f'📊 Alert Evidence IPs loaded (14d): {alert_evidence_df.count()}')

# Device entities — join on DeviceId to match graph Device nodes
alert_evidence_device_df = (alert_evidence_raw
 .where(
 (col('EntityType') == 'Machine') &
 col('DeviceId').isNotNull() & (col('DeviceId') != '')
 )
 .select('AlertId', 'DeviceId', 'EvidenceRole', 'TimeGenerated')
)
print(f'📊 Alert Evidence Devices loaded (14d): {alert_evidence_device_df.count()}')

# User entities — use AccountObjectId to match graph User nodes (UserId = AccountObjectId)
alert_evidence_user_df = (alert_evidence_raw
 .where(
 (col('EntityType') == 'User') &
 col('AccountObjectId').isNotNull() & (col('AccountObjectId') != '')
 )
 .select('AlertId', col('AccountObjectId').alias('UserId'), 'EvidenceRole', 'TimeGenerated')
)
print(f'📊 Alert Evidence Users loaded (14d): {alert_evidence_user_df.count()}')

# --- Threat intelligence indicators — active, non-revoked IP indicators (30-day window) ---
ti_df = lake_provider.read_table('ThreatIntelIndicators', workspace_name)
ti_df = ti_df.df if hasattr(ti_df, 'df') else ti_df
ti_df = (ti_df
 .where(
 (col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS")) &
 (col('IsActive') == True) & (col('Revoked') == False) &
 ((col('ObservableKey') == 'network-traffic:src_ref.value') |
 (col('ObservableKey') == 'ipv4-addr:value'))
 )
 .select('Id', 'ObservableKey', 'ObservableValue', 'Confidence',
 'Tags', 'ValidFrom', 'ValidUntil', 'TimeGenerated')
).persist()
print(f'📊 TI IP indicators loaded (30d, active): {ti_df.count()}')

# --- UEBA behavioral analytics — ML-based investigation priority scoring by IP ---
ueba_df = lake_provider.read_table('BehaviorAnalytics', workspace_name)
ueba_df = ueba_df.df if hasattr(ueba_df, 'df') else ueba_df
ueba_df = (ueba_df
 .where(
 (col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS")) &
 col('SourceIPAddress').isNotNull() & (col('SourceIPAddress') != '')
 )
 .select('SourceIPAddress', 'SourceIPLocation', 'InvestigationPriority', 'TimeGenerated')
).persist()
print(f'📊 UEBA records loaded (14d): {ueba_df.count()}')


In [ ]:
# =============================================================================
# STEP 2: Parse MITRE fields from SentinelBehaviorInfo
# - Categories: '["Discovery"]' or '["DefenseEvasion","Discovery"]' -> Tactics
# - AttackTechniques: '["Cloud Service Discovery (T1526)"]' -> Techniques
# =============================================================================

# Parse JSON arrays for tactics and techniques
parsed_behavior_df = (behavior_df
 # Parse Categories (Tactics) - stored as JSON array string
 .withColumn('TacticArray',
 from_json(col('Categories'), ArrayType(StringType()))
 )
 # Parse AttackTechniques - stored as JSON array string
 .withColumn('TechniqueArray',
 from_json(col('AttackTechniques'), ArrayType(StringType()))
 )
 # Keep needed columns
 .select(
 'BehaviorId',
 'Title',
 'ActionType',
 'Description',
 'StartTime',
 'EndTime',
 'TimeGenerated',
 'ServiceSource',
 'DetectionSource',
 'TacticArray',
 'TechniqueArray'
 )
).persist()

print(f'📊 Parsed behaviors: {parsed_behavior_df.count()}')
parsed_behavior_df.show(5, truncate=50)

## Data Quality Check

Verify distributions and data quality before building graph nodes and edges.


In [ ]:
# =============================================================================
# Data Quality Check — verify all source tables have expected distributions
# =============================================================================

print('🔍 === Distinct Tactics ===')
parsed_behavior_df.select(explode('TacticArray').alias('tactic')).distinct().show(truncate=False)

print('🔍 === Distinct Techniques (sample) ===')
parsed_behavior_df.select(explode('TechniqueArray').alias('technique')).distinct().show(10, truncate=False)

print('🔍 === Entity Types in Entities Table ===')
entities_df.groupBy('EntityType', 'EntityRole').count().orderBy(col('count').desc()).show()

print('🔍 === Sample IPs ===')
entities_df.where(col('EntityType') == 'Ip').select('RemoteIP', 'EntityRole').distinct().show(10)

print('🔍 === Alert Severity Distribution ===')
alert_info_df.groupBy('Severity').count().orderBy(col('count').desc()).show()

print('🔍 === TI Observable Type Distribution ===')
ti_df.groupBy('ObservableKey').count().show()

print('🔍 === UEBA Investigation Priority Distribution ===')
ueba_df.groupBy('InvestigationPriority').count().orderBy(col('count').desc()).show()

## Node DataFrames

Extract and reshape source data into typed node DataFrames.


In [ ]:
# =============================================================================
# STEP 3: Extract NODE DataFrames
# 9 node types: Behavior, Tactic, Technique, IP, User, AWSResource, Device,
# Alert, ThreatIndicator
# =============================================================================

# --- Extract context properties from entities table (enrichments for Behavior nodes) ---
session_props_df = (entities_df
 .where(col('EntityType') == 'CloudLogonSession')
 .select(
 col('BehaviorId'),
 get_json_object(col('AdditionalFields'), '$.SessionId').alias('SessionId')
 )
 .where(col('SessionId').isNotNull())
 .groupBy('BehaviorId')
 .agg(first('SessionId').alias('SessionId'))
)

security_group_props_df = (entities_df
 .where(col('EntityType') == 'SecurityGroup')
 .select(
 col('BehaviorId'),
 get_json_object(col('AdditionalFields'), '$.SID').alias('SecurityGroupId')
 )
 .where(col('SecurityGroupId').isNotNull())
 .groupBy('BehaviorId')
 .agg(first('SecurityGroupId').alias('SecurityGroupId'))
)

cloud_app_props_df = (entities_df
 .where(col('EntityType') == 'CloudApplication')
 .select(
 col('BehaviorId'),
 col('ApplicationId').alias('CloudAppId'),
 col('Application').alias('CloudAppName')
 )
 .groupBy('BehaviorId')
 .agg(
 first('CloudAppId').alias('CloudAppId'),
 first('CloudAppName').alias('CloudAppName')
 )
)

# --- Behavior Nodes ---
# Core behavioral detection with temporal analysis properties for off-hours/weekend detection
behavior_nodes_df = (parsed_behavior_df
 .select(
 col('BehaviorId'), col('Title'), col('ActionType'),
 col('Description'), col('StartTime'), col('EndTime'),
 col('ServiceSource')
 )
 .distinct()
 .where(col('BehaviorId').isNotNull())
 .join(session_props_df, 'BehaviorId', 'left')
 .join(security_group_props_df, 'BehaviorId', 'left')
 .join(cloud_app_props_df, 'BehaviorId', 'left')
 # Temporal analysis properties — enable off-hours and weekend anomaly detection
 .withColumn('HourOfDay', hour(col('StartTime')))
 .withColumn('DayOfWeek', dayofweek(col('StartTime')))
 .withColumn('IsWeekend',
 when(dayofweek(col('StartTime')).isin(1, 7), True).otherwise(False))
)

# --- Tactic Nodes ---
# MITRE ATT&CK tactics extracted from behavior Categories field
tactic_nodes_df = (parsed_behavior_df
 .select(explode(col('TacticArray')).alias('TacticId'))
 .select(trim(col('TacticId')).alias('TacticId'))
 .distinct()
 .where(col('TacticId').isNotNull() & (col('TacticId') != ''))
)

# --- Technique Nodes ---
# MITRE ATT&CK techniques extracted from behavior AttackTechniques field
technique_nodes_df = (parsed_behavior_df
 .select(explode(col('TechniqueArray')).alias('TechniqueId'))
 .select(trim(col('TechniqueId')).alias('TechniqueId'))
 .distinct()
 .where(col('TechniqueId').isNotNull() & (col('TechniqueId') != ''))
)

# --- IP Nodes ---
# Enriched with UEBA risk scoring (InvestigationPriority, event count, geolocation)
ip_base_df = (entities_df
 .where(col('EntityType') == 'Ip')
 .select(col('RemoteIP').alias('IPAddress'))
 .distinct()
 .where(col('IPAddress').isNotNull() & (col('IPAddress') != ''))
)

# Aggregate UEBA risk scores per IP — max priority indicates highest ML-assessed risk
ueba_agg_df = (ueba_df
 .groupBy('SourceIPAddress')
 .agg(
 F.max('InvestigationPriority').alias('UEBAMaxPriority'),
 F.count('*').alias('UEBAEventCount'),
 first('SourceIPLocation').alias('IPLocation')
 )
)

ip_nodes_df = (ip_base_df
 .join(ueba_agg_df, ip_base_df.IPAddress == ueba_agg_df.SourceIPAddress, 'left')
 .select(
 ip_base_df.IPAddress,
 coalesce(col('UEBAMaxPriority'), lit(0)).alias('UEBAMaxPriority'),
 coalesce(col('UEBAEventCount'), lit(0)).alias('UEBAEventCount'),
 col('IPLocation')
 )
)

# --- User Nodes ---
user_nodes_df = (entities_df
 .where(col('EntityType') == 'User')
 .select(
 col('AccountObjectId').alias('UserId'),
 col('AccountName')
 )
 .distinct()
 .where(col('UserId').isNotNull() & (col('UserId') != ''))
)

# --- AWS Resource Nodes ---
# Cloud resources targeted by behaviors (EC2, S3, IAM, EKS, Secrets Manager, etc.)
aws_resource_nodes_df = (entities_df
 .where(col('EntityType') == 'AmazonResource')
 .select(
 col('CloudResourceId'),
 col('CloudPlatform'),
 col('CloudResourceType')
 )
 .distinct()
 .where(col('CloudResourceId').isNotNull() & (col('CloudResourceId') != ''))
)

# --- Device Nodes ---
device_nodes_df = (entities_df
 .where((col('EntityType') == 'Host') | (col('EntityType') == 'Device'))
 .select(col('DeviceId'), col('DeviceName'))
 .distinct()
 .where(col('DeviceId').isNotNull() | col('DeviceName').isNotNull())
)

# --- Alert Nodes ---
# Security alerts from Defender products, filtered to alerts involving behavior-linked IPs
# Include alerts linked to IPs, Devices, or Users involved in behaviors
relevant_alert_ids_ip = (alert_evidence_df
 .join(ip_base_df, alert_evidence_df.RemoteIP == ip_base_df.IPAddress, 'inner')
 .select(alert_evidence_df.AlertId)
)
relevant_alert_ids_device = (alert_evidence_device_df
 .join(device_nodes_df, 'DeviceId', 'inner')
 .select('AlertId')
)
relevant_alert_ids_user = (alert_evidence_user_df
 .join(user_nodes_df, 'UserId', 'inner')
 .select('AlertId')
)
relevant_alert_ids = relevant_alert_ids_ip.union(relevant_alert_ids_device).union(relevant_alert_ids_user).distinct()
alert_nodes_df = (alert_info_df
 .join(relevant_alert_ids, 'AlertId', 'inner')
 .select('AlertId', 'Title', 'Severity', 'Category',
 'ServiceSource', 'DetectionSource')
 .distinct()
)

# --- ThreatIndicator Nodes ---
# Active threat intelligence IP indicators matched against behavior IPs
# Deduplicate by indicator Id — keep most recent ingestion per indicator
ti_window = Window.partitionBy('Id').orderBy(col('TimeGenerated').desc())
ti_deduped_df = (ti_df
 .withColumn('rn', row_number().over(ti_window))
 .where(col('rn') == 1)
 .drop('rn')
)

# Inner join TI with behavior IPs — only include indicators that match observed IPs
ti_matched_df = (ti_deduped_df
 .join(ip_base_df, ti_deduped_df.ObservableValue == ip_base_df.IPAddress, 'inner')
)

ti_nodes_df = (ti_matched_df
 .select(
 col('Id'),
 col('ObservableValue'),
 col('ObservableKey'),
 col('Confidence'),
 col('Tags'),
 col('ValidFrom'),
 col('ValidUntil')
 )
 .distinct()
)

# --- Print all node counts ---
print(f'📊 Nodes - Behaviors: {behavior_nodes_df.count()}')
print(f'📊 Nodes - Tactics: {tactic_nodes_df.count()}')
print(f'📊 Nodes - Techniques: {technique_nodes_df.count()}')
print(f'📊 Nodes - IPs: {ip_nodes_df.count()}')
print(f'📊 Nodes - Users: {user_nodes_df.count()}')
print(f'📊 Nodes - AWS Resources: {aws_resource_nodes_df.count()}')
print(f'📊 Nodes - Devices: {device_nodes_df.count()}')
print(f'📊 Nodes - Alerts: {alert_nodes_df.count()}')
print(f'📊 Nodes - ThreatIndicators: {ti_nodes_df.count()}')


## Edge DataFrames

Build edge DataFrames that connect nodes through relationship types.


In [ ]:
# =============================================================================
# STEP 4: Extract EDGE DataFrames (8 edge types)
# All edges are deduplicated via EdgeKey to prevent duplicate relationships
# =============================================================================

# --- Behavior -> Tactic edges ---
# Maps each behavior to its MITRE ATT&CK tactic(s)
behavior_tactic_edges_df = (parsed_behavior_df
 .select(
 col('BehaviorId'),
 explode(col('TacticArray')).alias('TacticId'),
 col('TimeGenerated'),
 col('StartTime')
 )
 .select(
 col('BehaviorId'),
 trim(col('TacticId')).alias('TacticId'),
 col('TimeGenerated'),
 col('StartTime')
 )
 .where(col('BehaviorId').isNotNull() & col('TacticId').isNotNull() & (col('TacticId') != ''))
 .withColumn('EdgeKey', concat_ws('|', col('BehaviorId'), col('TacticId')))
 .dropDuplicates(['EdgeKey'])
)

# --- Behavior -> Technique edges ---
# Maps each behavior to its MITRE ATT&CK technique(s)
behavior_technique_edges_df = (parsed_behavior_df
 .select(
 col('BehaviorId'),
 explode(col('TechniqueArray')).alias('TechniqueId'),
 col('TimeGenerated')
 )
 .select(
 col('BehaviorId'),
 trim(col('TechniqueId')).alias('TechniqueId'),
 col('TimeGenerated')
 )
 .where(col('BehaviorId').isNotNull() & col('TechniqueId').isNotNull() & (col('TechniqueId') != ''))
 .withColumn('EdgeKey', concat_ws('|', col('BehaviorId'), col('TechniqueId')))
 .dropDuplicates(['EdgeKey'])
)

# --- IP -> Behavior edges ---
# Links IP addresses to the behaviors they participated in
ip_behavior_edges_df = (entities_df
 .where((col('EntityType') == 'Ip') & col('RemoteIP').isNotNull() & (col('RemoteIP') != ''))
 .select(
 col('RemoteIP').alias('IPAddress'),
 col('BehaviorId'),
 col('EntityRole'),
 col('DetailedEntityRole'),
 col('TimeGenerated')
 )
 .where(col('BehaviorId').isNotNull())
 .withColumn('EdgeKey', concat_ws('|', col('IPAddress'), col('BehaviorId')))
 .dropDuplicates(['EdgeKey'])
)

# --- User -> Behavior edges ---
# Links user identities to the behaviors they performed
user_behavior_edges_df = (entities_df
 .where((col('EntityType') == 'User') & col('AccountObjectId').isNotNull() & (col('AccountObjectId') != ''))
 .select(
 col('AccountObjectId').alias('UserId'),
 col('BehaviorId'),
 col('EntityRole'),
 col('DetailedEntityRole'),
 col('TimeGenerated')
 )
 .where(col('BehaviorId').isNotNull())
 .withColumn('EdgeKey', concat_ws('|', col('UserId'), col('BehaviorId')))
 .dropDuplicates(['EdgeKey'])
)

# --- AWS Resource -> Behavior edges ---
# Links AWS cloud resources to the behaviors targeting them
aws_behavior_edges_df = (entities_df
 .where((col('EntityType') == 'AmazonResource') & col('CloudResourceId').isNotNull() & (col('CloudResourceId') != ''))
 .select(
 col('CloudResourceId'),
 col('BehaviorId'),
 col('EntityRole'),
 col('DetailedEntityRole'),
 col('TimeGenerated')
 )
 .where(col('BehaviorId').isNotNull())
 .withColumn('EdgeKey', concat_ws('|', col('CloudResourceId'), col('BehaviorId')))
 .dropDuplicates(['EdgeKey'])
)

# --- Device -> Behavior edges ---
# Links devices/hosts to their associated behaviors
device_behavior_edges_df = (entities_df
 .where(((col('EntityType') == 'Host') | (col('EntityType') == 'Device'))
 & (col('DeviceId').isNotNull() | col('DeviceName').isNotNull()))
 .select(
 coalesce(col('DeviceId'), col('DeviceName')).alias('DeviceKey'),
 col('BehaviorId'),
 col('EntityRole'),
 col('DetailedEntityRole'),
 col('TimeGenerated')
 )
 .where(col('BehaviorId').isNotNull())
 .withColumn('EdgeKey', concat_ws('|', col('DeviceKey'), col('BehaviorId')))
 .dropDuplicates(['EdgeKey'])
)

# --- IP -> Alert edges ---
# Links IPs to the security alerts where they appeared as evidence (via AlertEvidence)
ip_triggered_alert_edges_df = (alert_evidence_df
 .join(ip_base_df, alert_evidence_df.RemoteIP == ip_base_df.IPAddress, 'inner')
 .select(
 ip_base_df.IPAddress,
 alert_evidence_df.AlertId,
 alert_evidence_df.EvidenceRole,
 alert_evidence_df.TimeGenerated
 )
 .withColumn('EdgeKey', concat_ws('|', col('IPAddress'), col('AlertId')))
 .dropDuplicates(['EdgeKey'])
)

# --- Device -> Alert edges ---
device_triggered_alert_edges_df = (alert_evidence_device_df
 .join(device_nodes_df, 'DeviceId', 'inner')
 .select(
 col('DeviceId'),
 col('AlertId'),
 col('EvidenceRole'),
 col('TimeGenerated')
 )
 .withColumn('EdgeKey', concat_ws('|', col('DeviceId'), col('AlertId')))
 .dropDuplicates(['EdgeKey'])
)

# --- User -> Alert edges ---
user_triggered_alert_edges_df = (alert_evidence_user_df
 .join(user_nodes_df, 'UserId', 'inner')
 .select(
 col('UserId'),
 col('AlertId'),
 col('EvidenceRole'),
 col('TimeGenerated')
 )
 .withColumn('EdgeKey', concat_ws('|', col('UserId'), col('AlertId')))
 .dropDuplicates(['EdgeKey'])
)

# --- IP -> ThreatIndicator edges ---
# Links IPs to matching threat intelligence indicators
ip_matches_ti_edges_df = (ti_matched_df
 .select(
 col('IPAddress'),
 col('Id'),
 col('Confidence'),
 ti_matched_df.TimeGenerated
 )
 .withColumn('EdgeKey', concat_ws('|', col('IPAddress'), col('Id')))
 .dropDuplicates(['EdgeKey'])
)

# --- Print all edge counts ---
print(f'📊 Edges - BehaviorMappedToTactic: {behavior_tactic_edges_df.count()}')
print(f'📊 Edges - BehaviorMappedToTechnique: {behavior_technique_edges_df.count()}')
print(f'📊 Edges - BehaviorInvolvesIP: {ip_behavior_edges_df.count()}')
print(f'📊 Edges - BehaviorInvolvesUser: {user_behavior_edges_df.count()}')
print(f'📊 Edges - BehaviorInvolvesAWSResource: {aws_behavior_edges_df.count()}')
print(f'📊 Edges - BehaviorInvolvesDevice: {device_behavior_edges_df.count()}')
print(f'📊 Edges - IPTriggeredAlert: {ip_triggered_alert_edges_df.count()}')
print(f'📊 Edges - DeviceTriggeredAlert: {device_triggered_alert_edges_df.count()}')
print(f'📊 Edges - UserTriggeredAlert: {user_triggered_alert_edges_df.count()}')
print(f'📊 Edges - IPMatchesTI: {ip_matches_ti_edges_df.count()}')


## Graph Assembly

Assembling the graph with the `GraphSpecBuilder` API:
- **9 Node Types** -- Behavior, Tactic, Technique, IP (UEBA-enriched), User, AWSResource, Device, Alert, ThreatIndicator
- **10 Edge Types** -- 6 entity-behavior relationships + IP-Alert correlation + IP-ThreatIntel matching

The IP node serves as the central hub connecting behavioral detections, security alerts, and threat intelligence -- enabling multi-hop traversals that span all three detection methodologies in a single GQL query.

In [ ]:
# =============================================================================
# STEP 5: Build Graph (9 nodes, 10 edges)
# =============================================================================

behavior_attack_chain_graph = (GraphSpecBuilder.start()

 # ===================== NODES (9 types) =====================
 .add_node('Behavior')
 .from_dataframe(behavior_nodes_df)
 .with_columns('BehaviorId', 'Title', 'ActionType', 'Description',
 'StartTime', 'EndTime', 'ServiceSource',
 'SessionId', 'SecurityGroupId', 'CloudAppId', 'CloudAppName',
 'HourOfDay', 'DayOfWeek', 'IsWeekend',
 key='BehaviorId', display='Title')

 .add_node('Tactic')
 .from_dataframe(tactic_nodes_df)
 .with_columns('TacticId', key='TacticId', display='TacticId')

 .add_node('Technique')
 .from_dataframe(technique_nodes_df)
 .with_columns('TechniqueId', key='TechniqueId', display='TechniqueId')

 .add_node('IP')
 .from_dataframe(ip_nodes_df)
 .with_columns('IPAddress', 'UEBAMaxPriority', 'UEBAEventCount', 'IPLocation',
 key='IPAddress', display='IPAddress')

 .add_node('User')
 .from_dataframe(user_nodes_df)
 .with_columns('UserId', 'AccountName', key='UserId', display='AccountName')

 .add_node('AWSResource')
 .from_dataframe(aws_resource_nodes_df)
 .with_columns('CloudResourceId', 'CloudPlatform', 'CloudResourceType',
 key='CloudResourceId', display='CloudResourceId')

 .add_node('Device')
 .from_dataframe(device_nodes_df)
 .with_columns('DeviceId', 'DeviceName',
 key='DeviceId', display='DeviceName')

 .add_node('Alert')
 .from_dataframe(alert_nodes_df)
 .with_columns('AlertId', 'Title', 'Severity', 'Category',
 'ServiceSource', 'DetectionSource',
 key='AlertId', display='Title')

 .add_node('ThreatIndicator')
 .from_dataframe(ti_nodes_df)
 .with_columns('Id', 'ObservableValue', 'ObservableKey',
 'Confidence', 'Tags', 'ValidFrom', 'ValidUntil',
 key='Id', display='ObservableValue')

 # ===================== EDGES (10 types) =====================
 .add_edge('BehaviorMappedToTactic')
 .from_dataframe(behavior_tactic_edges_df)
 .source(id_column='BehaviorId', node_type='Behavior')
 .target(id_column='TacticId', node_type='Tactic')
 .with_columns('TimeGenerated', 'StartTime', 'EdgeKey', key='EdgeKey', display='EdgeKey')

 .add_edge('BehaviorMappedToTechnique')
 .from_dataframe(behavior_technique_edges_df)
 .source(id_column='BehaviorId', node_type='Behavior')
 .target(id_column='TechniqueId', node_type='Technique')
 .with_columns('TimeGenerated', 'EdgeKey', key='EdgeKey', display='EdgeKey')

 .add_edge('BehaviorInvolvesIP')
 .from_dataframe(ip_behavior_edges_df)
 .source(id_column='BehaviorId', node_type='Behavior')
 .target(id_column='IPAddress', node_type='IP')
 .with_columns('EntityRole', 'DetailedEntityRole', 'TimeGenerated', 'EdgeKey',
 key='EdgeKey', display='EdgeKey')

 .add_edge('BehaviorInvolvesUser')
 .from_dataframe(user_behavior_edges_df)
 .source(id_column='BehaviorId', node_type='Behavior')
 .target(id_column='UserId', node_type='User')
 .with_columns('EntityRole', 'DetailedEntityRole', 'TimeGenerated', 'EdgeKey',
 key='EdgeKey', display='EdgeKey')

 .add_edge('BehaviorInvolvesAWSResource')
 .from_dataframe(aws_behavior_edges_df)
 .source(id_column='BehaviorId', node_type='Behavior')
 .target(id_column='CloudResourceId', node_type='AWSResource')
 .with_columns('EntityRole', 'DetailedEntityRole', 'TimeGenerated', 'EdgeKey',
 key='EdgeKey', display='EdgeKey')

 .add_edge('BehaviorInvolvesDevice')
 .from_dataframe(device_behavior_edges_df)
 .source(id_column='BehaviorId', node_type='Behavior')
 .target(id_column='DeviceKey', node_type='Device')
 .with_columns('EntityRole', 'DetailedEntityRole', 'TimeGenerated', 'EdgeKey',
 key='EdgeKey', display='EdgeKey')

 .add_edge('IPTriggeredAlert')
 .from_dataframe(ip_triggered_alert_edges_df)
 .source(id_column='IPAddress', node_type='IP')
 .target(id_column='AlertId', node_type='Alert')
 .with_columns('EvidenceRole', 'TimeGenerated', 'EdgeKey',
 key='EdgeKey', display='EdgeKey')

 .add_edge('DeviceTriggeredAlert')
 .from_dataframe(device_triggered_alert_edges_df)
 .source(id_column='DeviceId', node_type='Device')
 .target(id_column='AlertId', node_type='Alert')
 .with_columns('EvidenceRole', 'TimeGenerated', 'EdgeKey',
 key='EdgeKey', display='EdgeKey')

 .add_edge('UserTriggeredAlert')
 .from_dataframe(user_triggered_alert_edges_df)
 .source(id_column='UserId', node_type='User')
 .target(id_column='AlertId', node_type='Alert')
 .with_columns('EvidenceRole', 'TimeGenerated', 'EdgeKey',
 key='EdgeKey', display='EdgeKey')

 .add_edge('IPMatchesTI')
 .from_dataframe(ip_matches_ti_edges_df)
 .source(id_column='IPAddress', node_type='IP')
 .target(id_column='Id', node_type='ThreatIndicator')
 .with_columns('Confidence', 'TimeGenerated', 'EdgeKey',
 key='EdgeKey', display='EdgeKey')

).done()



In [ ]:
# Check the schema of the graph spec to ensure it is correct
behavior_attack_chain_graph.show_schema()


## Build & Publish

Build the graph from the spec. This validates the schema, prepares the data, and publishes the graph for querying.


In [ ]:
# Build the graph from the spec - this will validate the spec and prepare it for querying
# Alternative: use Graph.prepare() to prepare the graph nodes and edges in the lake
# and then use Graph.publish() to create the graph. You would typically call prepare()
# and publish() separately to understand the cost of Graph API calls triggered by Graph.publish()
# see https://learn.microsoft.com/azure/sentinel/billing?tabs=simplified%2Ccommitment-tiers
behavior_graph = Graph.build(behavior_attack_chain_graph)
